# 🚀 Notebook do Professor (Demo) — Aula 03: Structured Output e Pydantic v2

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 03/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🧱 BaseModel · PydanticOutputParser**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Fazer o LLM retornar dados num schema garantido pelo Pydantic — não em texto livre. Ao final, a chain do grupo aceita um input e retorna um objeto Python com campos validados e tipados.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 06 — BaseModel — definindo um schema em Python

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
!pip install pydantic -q  # já vem com langchain, mas explicitando

from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

# Schema: cada campo tem tipo e descrição
class Curriculo(BaseModel):
    nome:         str               = Field(description="Nome completo do candidato")
    email:        str               = Field(description="E-mail do candidato")
    habilidades:  List[str]         = Field(description="Lista de habilidades técnicas")
    anos_exp:     int               = Field(description="Anos de experiência profissional")
    senioridade:  Optional[str]    = Field(None, description="junior/pleno/senior")

# Instanciar — Pydantic valida os tipos automaticamente
curriculo = Curriculo(
    nome="Ana Silva",
    email="ana@email.com",
    habilidades=["Python", "LangChain"],
    anos_exp=3,
)
print(curriculo.nome)        # → "Ana Silva"
print(curriculo.habilidades)  # → ["Python", "LangChain"]
print(curriculo.model_dump())  # → dict Python (Pydantic v2)
print(curriculo.model_dump_json())  # → string JSON

### Slide 07 — ValidationError — o Pydantic detecta dados inválidos

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import List, Literal

class Curriculo(BaseModel):
    nome:        str
    anos_exp:    int                           # Deve ser inteiro
    senioridade: Literal["junior","pleno","senior"]  # Enum implícito
    habilidades: List[str]

    # Validator customizado — Pydantic v2
    @field_validator("anos_exp")
    @classmethod
    def anos_positivos(cls, v):
        if v < 0:
            raise ValueError("anos_exp deve ser positivo")
        return v

# Teste de falha — tratar com try/except
try:
    c = Curriculo(
        nome="Carlos",
        anos_exp="dez",         # ❌ str onde int era esperado
        senioridade="ninja",    # ❌ valor fora do Literal
        habilidades=["Python"],
    )
except ValidationError as e:
    print(f"Erros encontrados: {e.error_count()}")
    for erro in e.errors():
        print(f"  Campo: {erro['loc']} | Erro: {erro['msg']}")
    # → Campo: ('anos_exp',) | Erro: Input should be a valid integer
    # → Campo: ('senioridade',) | Erro: Input should be 'junior', 'pleno' or 'senior'

### Slide 09 — PydanticOutputParser — schema vira instrução para o modelo

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

class Curriculo(BaseModel):
    nome:        str       = Field(description="Nome completo")
    email:       str       = Field(description="Endereço de e-mail")
    habilidades: List[str] = Field(description="Habilidades técnicas listadas")
    anos_exp:    int       = Field(description="Anos de experiência")

# 1. Criar o parser com o schema
parser = PydanticOutputParser(pydantic_object=Curriculo)

# 2. O parser gera as instruções de formato automaticamente
print(parser.get_format_instructions())
# → "Return a JSON object with the following schema: {nome: ..., email: ...}"

# 3. Injetar as instruções no prompt com {format_instructions}
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um extrator de dados de currículos.\n{format_instructions}"),
    ("human",  "Extraia as informações deste currículo:\n{curriculo}"),
]).partial(format_instructions=parser.get_format_instructions())

# 4. Chain completa com PydanticOutputParser no final
chain = prompt | llm | parser

# 5. Invocar — retorna objeto Pydantic, não string
resultado = chain.invoke({"curriculo": "João Silva, joao@email.com, 5 anos..."})
print(type(resultado))      # → <class 'Curriculo'>
print(resultado.nome)        # → "João Silva"
print(resultado.anos_exp)    # → 5  (int, não string!)

### Slide 10 — O método .partial() — pré-preenchendo variáveis

In [ ]:
# Precisa passar format_instructions em todo invoke
chain.invoke({
    "curriculo": "texto...",
    "format_instructions":
        parser.get_format_instructions(),
})
# repetitivo e propenso a erros de esquecimento

### Slide 10 — O método .partial() — pré-preenchendo variáveis

```
# .partial() fixa a variável no template
prompt = ChatPromptTemplate...partial(
    format_instructions=
        parser.get_format_instructions(),
)

# Agora invoke só precisa das variáveis dinâmicas
chain.invoke({"curriculo": "texto..."})
```

### Slide 11 — format="json" nativo do Ollama — alternativa direta

In [ ]:
# Opção 1: format="json" no ChatOllama
# Garante que o modelo retorne JSON válido — mas sem validar o schema
llm_json = ChatOllama(
    model="gpt-oss:120b",
    format="json",   # Ollama garante JSON válido na saída
)

# Com JsonOutputParser — parseia o JSON mas não valida schema
chain_json = prompt | llm_json | JsonOutputParser()
resultado = chain_json.invoke({"curriculo": "texto..."})
print(type(resultado))  # → dict (sem validação de campos)

# Opção 2: format="json" + PydanticOutputParser (combinação recomendada)
llm_json = ChatOllama(model="gpt-oss:120b", format="json")
chain_validado = prompt | llm_json | parser   # parser = PydanticOutputParser
resultado = chain_validado.invoke({"curriculo": "texto..."})
print(type(resultado))  # → <class 'Curriculo'> (objeto Pydantic validado)

### Slide 12 — Schemas avançados — tipos Python úteis para LLMs

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Dict

# Caso 1: Classificador de tickets de suporte
class Ticket(BaseModel):
    categoria:  Literal["bug", "feature", "duvida"]
    urgencia:   Literal["baixa", "media", "alta"]
    resumo:     str     = Field(max_length=100)  # limite de caracteres
    confianca:  float   = Field(ge=0, le=1)  # entre 0 e 1

# Caso 2: Gerador de relatório com objetos aninhados
class Ponto(BaseModel):
    titulo:    str
    descricao: str

class Relatorio(BaseModel):
    titulo:        str
    pontos_fortes: List[Ponto]  # lista de objetos aninhados
    pontos_fracos: List[Ponto]
    nota_final:    float = Field(ge=0, le=10)

# Caso 3: Dict com chaves dinâmicas (quando o schema varia)
class Entidades(BaseModel):
    pessoas:    List[str]
    empresas:   List[str]
    localizacoes: List[str]

### Slide 15 — Integrar structured output com a memória da Aula 02

In [ ]:
# Estratégia: 2 chains separadas na mesma aplicação
# Chain 1: conversa normal com memória (para diálogo)
# Chain 2: extração com Pydantic (para quando precisar de dados estruturados)

# Chain de conversa (Aula 02)
chat = ConversationChain(llm=llm, memory=memoria, prompt=prompt_custom)

# Chain de extração (Aula 03)
chain_extrator = prompt_pydantic | llm_json | parser

def processar_consulta(entrada: str, modo: str = "chat"):
    if modo == "chat":
        # Diálogo normal com memória
        return chat.predict(input=entrada)
    elif modo == "extrair":
        # Extração estruturada (sem memória — cada extração é independente)
        return chain_extrator.invoke({"pergunta": entrada})

# Uso: usuário conversa normalmente, mas pode pedir extração
resposta_chat = processar_consulta("Me fale sobre feijoada", "chat")
receita_obj   = processar_consulta("Crie a receita de feijoada", "extrair")
print(receita_obj.model_dump())  # → dict com campos validados

### Slide 21 — Python novo desta aula

In [ ]:
# 1. Herança de classe — BaseModel é a classe pai
class MinhaClasse(BaseModel):  # herda validação automática
    campo: str

# 2. Type hints avançados
from typing import List, Optional, Literal
campo1: List[str]              # lista de strings
campo2: Optional[int] = None   # pode ser None
campo3: Literal["a","b"]       # enum implícito — só "a" ou "b"

# 3. Decorador @field_validator (Pydantic v2)
@field_validator("campo")       # roda após o campo ser parseado
@classmethod
def validar(cls, v):
    if v < 0: raise ValueError("negativo")
    return v

# 4. .model_dump() e .model_dump_json() — Pydantic v2
obj.model_dump()       # → dict Python
obj.model_dump_json()  # → string JSON

# 5. .partial() em ChatPromptTemplate — pré-preencher variáveis
prompt.partial(variavel_fixa="valor")

# 6. try/except com tipo específico
try:
    resultado = chain.invoke(...)
except ValidationError as e:   # captura só ValidationError
    print(e.errors())            # lista de erros detalhados

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 03 · 2º Semestre**  
### Schema Pydantic para o domínio do grupo ★★

*Grupo 3–4 · 20 minutos · Google Colab*

1. Defina o schema com pelo menos 4 campos: mínimo 1 str, 1 numérico (int ou float), 1 List[str] e 1 Optional.
2. Complete as 4 lacunas — classe Pydantic, parser, prompt com .partial() e chain com try/except.
3. Teste com 2 inputs diferentes — um bem estruturado e um vago (ex: "não sei bem"). Observe como o modelo preenche campos opcionais.
4. Provoque um ValidationError : modifique o schema para incluir um Literal com valores específicos. Peça ao modelo algo fora dos valores aceitos e observe o erro capturado.

> **🎯 Orientação das lacunas (exemplo — domínio culinária)**
>
> Lacuna 1:  a classe do schema precisa de nome do prato, porções (numérico), lista de ingredientes, tempo de preparo (numérico) e um campo opcional de dificuldade restrito a 3 valores fixos.
>
> Lacuna 2:  o parser é instanciado passando a própria classe do schema como argumento.
>
> Lacuna 3:  o prompt define o papel de chef no system message, inclui a pergunta do usuário no human message, e fixa as instruções de formato do parser com .partial().
>
> Lacuna 4:  a chain conecta, nesta ordem, o prompt, o modelo e o parser.

> **💡 Dica:**
>
> as descriptions do Field são inseridas nas instruções que o parser gera — quanto mais claras, menor a chance do modelo errar o schema.

In [ ]:
!pip install langchain langchain-ollama pydantic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Literal
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 👉 LACUNA 1: defina a classe Pydantic para o domínio do grupo
# Mínimo 4 campos com tipos variados (str, int, List, Optional, Literal)
class NomeDaClasseDoGrupo(BaseModel):
    campo1: tipo = Field(description=___)
    campo2: tipo = Field(description=___)
    campo3: tipo = Field(description=___)
    campo4: Optional[tipo] = Field(None, description=___)

# 👉 LACUNA 2: crie o PydanticOutputParser com sua classe
parser = PydanticOutputParser(pydantic_object=___)

# 👉 LACUNA 3: crie o prompt com {format_instructions} e {pergunta}
# Use .partial() para pré-preencher format_instructions
prompt = ChatPromptTemplate.from_messages([
    ("system", ___),
    ("human",  ___),
]).partial(format_instructions=___)

llm   = ChatOllama(model="gpt-oss:120b", format="json")
# 👉 LACUNA 4: monte a chain e invoque com try/except
chain = ___ | ___ | ___

try:
    resultado = chain.invoke({"pergunta": ___})
    print(resultado.model_dump())
except ValidationError as e:
    print(f"Erro de schema: {e}")

## 📚 Referências da aula

- Docs Pydantic — BaseModel, Field, ValidationError (v2). docs.pydantic.dev/latest/concepts/models
- Docs LangChain — PydanticOutputParser: integração com Pydantic para saídas estruturadas. python.langchain.com/docs/modules/model_io/output_parsers/types/pydantic
- Docs LangChain — Structured output com with_structured_output() (abordagem alternativa moderna). python.langchain.com/docs/concepts/structured_outputs
- Docs Ollama — Structured outputs com format="json". ollama.com/blog/structured-outputs
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 22 — processamento de linguagem natural: a diferença entre extrair informação estruturada e gerar texto livre.

---

**→ Próxima Aula — Aula 04 · 24/08** — Context Engineering — de prompt engineering para context engineering
  
Gerenciar o contexto como recurso. CKP01 R4 + entrega.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*